In [ ]:
import pandas as pd
import gzip
import json
import os

# 1. دالة لقراءة ملفات الـ JSON المضغوطة
def load_amazon_data(file_path):
    print(f"جاري تحميل: {file_path}...") 
    data = [] 
    with gzip.open(file_path, 'rb') as f:      
       
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data) 

# ملاحظة: سنحمل فقط الأعمدة التي نحتاجها لتوفير الذاكرة
reviews_file = '/home/rwt/Documents/Movies_and_TV_5.json.gz'
meta_file = '/home/rwt/Documents/meta_Movies_and_TV.json.gz'


df_reviews = load_amazon_data(reviews_file)
df_reviews = df_reviews[['reviewerID', 'asin', 'overall', 'unixReviewTime']]

# تحميل (العناوين فقط)
df_meta = load_amazon_data(meta_file)
df_meta = df_meta[['asin', 'title']]

In [ ]:
#الدمج   
df_combined = pd.merge(df_reviews, df_meta, on='asin', how='inner')

# 4. التصفية للوصول لأرقام البحث (Sampling)
top_5000_items = df_combined['asin'].value_counts().nlargest(5000).index
df_filtered = df_combined[df_combined['asin'].isin(top_5000_items)]

In [ ]:
user_counts = df_filtered['reviewerID'].value_counts()
active_users = user_counts[user_counts >= 5].index
df_filtered = df_filtered[df_filtered['reviewerID'].isin(active_users)]


unique_users = df_filtered['reviewerID'].unique()
if len(unique_users) > 14386:
    import numpy as np
    selected_users = np.random.choice(unique_users, 14386, replace=False)
    df_filtered = df_filtered[df_filtered['reviewerID'].isin(selected_users)]

# 5. حفظ النتيجة النهائية
df_filtered.to_csv('amazon_movies_processed.csv', index=False)

print("--- Finished ---")
print(f"عدد المستخدمين: {df_filtered['reviewerID'].nunique()}")
print(f"عدد المنتجات: {df_filtered['asin'].nunique()}")
print(f"عدد التفاعلات: {len(df_filtered)}")